## Changelog
- parent: root
- change: end-to-end baseline pipeline — semantic NA encoding, basic temporal
        features, RandomForest on log-price with one-hot encoded categoricals
- hypothesis: establishes a defensible reference point. Treating "absent
        feature" NaNs as semantic zeros (rather than missing) and imputing
        only the genuinely missing values with leak-safe fitted lookups
        should outperform naive median/mode imputation across the board.
        OneHotEncoder fitted on train (handle_unknown='ignore') keeps
        train/test column geometry aligned and tolerates unseen test
        categories without crashing. log1p target matches the leaderboard
        metric.
- result: CV RMSE 0.1422 ± 0.0091 (5-fold, log-price)

## What the notebook does

The notebook runs a clean four-stage pipeline from raw CSVs to a Kaggle submission:

**Stage 1 — Setup.** 

**Stage 2 — NA encoding (`ames_na_encoding`).** A single `fit_imputers(train) → transform(train) → transform(test)` flow handles every missingness pattern in the dataset:
- Stateless absent-feature encodings for the 14+ columns where NaN means "feature doesn't exist" (basement, garage, pool, fence, alley, masonry veneer, etc.) — split into ordinal-with-zero, other ordinal scales, and nominal labels.
- Fitted imputers for genuine missingness: `LotFrontage` by neighborhood median, `Electrical` by training mode, six small-count categoricals by simple mode, `MSZoning` by conditional mode on `(MSSubClass, Neighborhood)`.

**Stage 3 — Feature engineering.** `add_temporal_features(drop_originals=True)` derives `HouseAge`, `YearsSinceRemod`, `WasRemodeled`, `IsNewHouse` and drops the raw `YearBuilt` / `YearRemodAdd` / `YrSold`.

**Stage 4 — Model (`ames_random_forest_baseline`).** Encodes the four mandatory quality columns (Ex/Gd/TA/Fa/Po → 5..1), `CentralAir` → 0/1, then `OneHotEncoder(handle_unknown='ignore')` on everything else fitted on train only. Targets `log1p(SalePrice)`. 5-fold CV with 300 trees, then refits on full train and predicts on test.

**Result: CV RMSE 0.1422 ± 0.0091 on log-price.** That's my RF baseline on Ames.

## What to do next, ranked by likely impact

I'd attack these in roughly this order. Each is its own changelog entry, so you can attribute gains to specific changes.

**1. Switch the model.** This is the single biggest lever and the cheapest to pull. Random Forest at 0.142 is solid, but on Ames a tuned gradient-boosted model typically lands at 0.115–0.125. Three options:

- `HistGradientBoostingRegressor` — sklearn-native, handles categoricals without one-hot, no extra dependency.
- `LightGBM` — faster, slightly better defaults, native categorical support via `categorical_feature=`.
- `XGBoost` — the Kaggle workhorse, requires one-hot or `enable_categorical=True`.

If you switch to one with native categorical support, you can also drop the one-hot expansion (~250 cols → ~80 cols) and let the model do better splits on high-cardinality columns like `Neighborhood`. Expect this single change to take you from 0.142 to roughly 0.125 with default hyperparameters.

**2. Add the standard Ames feature-engineering moves.** Each is small individually; together they typically buy 0.005–0.015 RMSE:

- `TotalSF = TotalBsmtSF + 1stFlrSF + 2ndFlrSF` — the single most predictive engineered feature on this dataset.
- `TotalLivingSF = GrLivArea + TotalBsmtSF`
- `TotalBath = FullBath + 0.5*HalfBath + BsmtFullBath + 0.5*BsmtHalfBath`
- `TotalPorchSF = WoodDeckSF + OpenPorchSF + EnclosedPorch + 3SsnPorch + ScreenPorch`
- Binary "has X" flags: `HasPool`, `HasGarage`, `HasBasement`, `HasFireplace`, `Has2ndFloor`.
- `OverallQual * GrLivArea` (or `* TotalSF`) — the quality-area interaction. Trees discover this implicitly but giving it to them directly often helps at the margin.

Add them to `ames_feature_engineering.py` as additional stateless functions (`add_area_features`, `add_bath_features`, `add_presence_flags`) so each can be toggled on/off in changelog experiments.

**3. Deal with outliers in train.** The Ames dataset has a few well-known outliers — most famously, four houses in Edwards neighborhood with `GrLivArea > 4000` and unusually low prices (these are the "partial sales" of unfinished homes that appear in `SaleCondition == 'Partial'`). Removing or down-weighting them is standard practice. A simple `train = train[train['GrLivArea'] < 4500]` typically drops CV RMSE by 0.005–0.01. It's the kind of thing where the ethics depend on whether your goal is leaderboard score (drop them) or honest model evaluation (keep them and let the model learn that segment).

**4. Better cross-validation.** Single 5-fold CV on 1460 rows has more variance than your `± 0.0091` suggests — that std is *across folds within one split*, not across different random splits. Use `RepeatedKFold(n_splits=5, n_repeats=3)` to get a more stable comparison number when you start A/B testing changes. The cost is 3x training time, but with RF/HGB on this dataset that's still under a minute.

**5. Robust target-aware encoding for high-cardinality categoricals.** `Neighborhood` has 25 levels; one-hot expands it to 25 columns where each carries a small sample. K-fold target encoding (mean `SalePrice` per neighborhood, computed out-of-fold to avoid leakage) typically shaves another 0.003–0.008 RMSE. This crosses into "more careful" territory and is easy to do wrong (leakage), so save it for after you've squeezed easier gains.

**6. Hyperparameter tuning.** Save this for last. Tuning RF or HGB on Ames typically yields ~0.005 RMSE improvement over defaults — real but small compared to gains 1–3 above. Use `optuna` or `HalvingGridSearchCV`. The reason to tune last: if you tune before improving features, you're optimizing a model that's about to be replaced, and you'll have to re-tune anyway.

## What not to do (yet)

- **Don't add polynomial features or huge interaction sets.** RF/GBM discover interactions on their own; you'd just add noise and slow training.
- **Don't stack/ensemble yet.** Stacking is for the last 0.005, after every individual model is well-tuned.
- **Don't over-engineer the imputation.** What you have is already careful; spending more time on `MSZoning`'s 4 missing rows is unlikely to move the needle.

## Suggested changelog for the next experiment

Filling in your template for the change I'd make first:

```
parent: root (CV RMSE 0.1422)
change: replace RandomForestRegressor with HistGradientBoostingRegressor,
        passing the categorical columns natively (skip OneHotEncoder)
hypothesis: gradient boosting has lower bias than RF on tabular regression,
        and native categorical handling avoids the information loss of
        one-hot on high-cardinality columns like Neighborhood. Expect
        CV RMSE to drop into the 0.12–0.13 range with default hyperparams.
```

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path

IS_KAGGLE = Path("/kaggle/input").exists()

if IS_KAGGLE:
    data_dir   = Path("/kaggle/input/home-data-for-ml-course")
    output_dir = Path("/kaggle/working")
else:
    def _find_competition_dir(start: Path) -> Path:
        for p in [start, *start.parents]:
            if (p / "config.yaml").exists() and (p / "data").is_dir():
                return p
        raise RuntimeError("competition dir not found (no ancestor has config.yaml + data/)")

    comp_dir   = _find_competition_dir(Path.cwd())
    data_dir   = comp_dir / "data"
    output_dir = Path.cwd()

for _p in [Path.cwd(), *Path.cwd().parents]:
    if (_p / "src" / "eda").is_dir():
        if str(_p) not in sys.path:
            sys.path.insert(0, str(_p))
        break

from src.eda import (
    numeric_columns,
    plot_histograms,
    plot_kde,
    plot_boxplots,
    plot_distributions,
    describe_df,
    get_missing_values,
    count_duplicates,
    plot_categorical_vs_target,
    plot_numerical_vs_target,
    target_rate_table,
    grouped_median,
)

train_data_raw = pd.read_csv(data_dir / "train.csv")
test_data_raw  = pd.read_csv(data_dir / "test.csv")


## NA Encoding

In [ ]:
assert (comp_dir / "utils" / "ames_na_encoding.py").exists(), \
    f"ames_na_encoding.py not found under {comp_dir / 'utils'}"

if str(comp_dir) not in sys.path:
    sys.path.insert(0, str(comp_dir))

from utils.ames_na_encoding import fit_imputers, transform

params = fit_imputers(train_data_raw)
train_data = transform(train_data_raw, params)
test_data  = transform(test_data_raw,  params)

## Feature Engineering

In [ ]:
from utils.ames_feature_engineering import add_temporal_features

train_data = add_temporal_features(train_data, drop_originals=True)
test_data  = add_temporal_features(test_data, drop_originals=True)

## Model

In [ ]:
from utils.ames_random_forest_baseline import fit_encoders, transform, train_random_forest

# Assumes train_data and test_data have already been through:
#   - encode_all_absent_as_zero / fit_imputers / transform (na_encoding)
#   - add_temporal_features (feature_engineering)

# 1. Fit encoders on train only
encoders = fit_encoders(train_data)

# 2. Transform both splits with the same encoders
train_X = transform(train_data.drop(columns=["SalePrice"]), encoders)
test_X  = transform(test_data, encoders)

# 3. Log-transform the target
train_y = np.log1p(train_data["SalePrice"])

# 4. Cross-validate + fit
rf = train_random_forest(train_X, train_y, n_estimators=300, cv_folds=5)

# 5. Predict on test (back-transform from log scale)
test_pred = np.expm1(rf.predict(test_X))

In [ ]:
# TODO: train model, compute predictions.

sample = pd.read_csv(data_dir / 'sample_submission.csv')
submission = sample.copy()
submission['SalePrice'] = test_pred
submission.to_csv(output_dir / "submission.csv", index=False)
